# Agentic RL customer-support reproduction

This notebook runs the public single-GPU case study: Base evaluation → LoRA SFT → evaluation → verifier-reward GRPO → final evaluation. The full-scale Qwen3-8B results in the README are historical results from the original experiment, not results produced by this notebook.

Recommended runtime: Google Colab A100. Expected public-run metrics are written only after the corresponding cells complete.

In [ ]:
!nvidia-smi
!git clone https://github.com/RongMu00/agentic-rl-customer-support-demo.git
%cd agentic-rl-customer-support-demo
!python -m pip install -q -r public_experiment/requirements-colab.txt

In [ ]:
# Fast contract checks before downloading or training a model.
!python -m public_experiment.generate_data
!python -m unittest discover -s tests -v

In [ ]:
# 1) Frozen base-model baseline on the held-out split.
!python -m public_experiment.evaluate --stage base --output results/public_base.json

In [ ]:
# 2) LoRA SFT and held-out evaluation.
!python -m public_experiment.train_sft
!python -m public_experiment.evaluate --stage sft --adapter artifacts/sft-adapter --output results/public_sft.json

In [ ]:
# 3) Model-generated rollout groups, sandbox/verifier rewards, and GRPO updates.
!python -m public_experiment.train_grpo
!python -m public_experiment.evaluate --stage grpo --adapter artifacts/grpo-adapter --output results/public_grpo.json

In [ ]:
# Compare only measurements generated by this notebook.
import json
import pandas as pd

rows = []
for stage in ('base', 'sft', 'grpo'):
    with open(f'results/public_{stage}.json') as stream:
        result = json.load(stream)
    rows.append({'stage': stage, **result['metrics']})
pd.DataFrame(rows).set_index('stage').T

## Interpretation

This is a deliberately smaller reproduction of the training and evaluation path. It is not evidence that the 0.5B public model is production-ready, and its metrics must not be substituted for the full-scale experiment results. Export the three JSON files when reporting the public run.